# Loading the datasets

In [155]:
#Importar librerias
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
pd.options.display.max_rows = 999
import warnings
warnings.filterwarnings("ignore")
import sys
sys.dont_write_bytecode = True

In [156]:
# Replace
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5\Data Tables\HDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['Today'] = datetime.today().date()
    MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
    MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
    MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
    MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    # data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    # data.columns = data.columns.get_level_values(level='id')
    return MasterDF
Replace = Loading_File(path)

In [157]:
# AIDE
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\AIDE T1D\Data Tables\AIDEDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['DateTime'] = MasterDF['DataDtTm']
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, errors='coerce', infer_datetime_format=True)
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    # data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    # data.columns = data.columns.get_level_values(level='id')
    return MasterDF
AIDE = Loading_File(path)

In [158]:
# Shanghai
from pathlib import Path
T1D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T1DM"
T2D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T2DM"
def Loading_File(file_path):
    files = list(Path(file_path).glob('*.xls*'))
    folder_dfs = pd.DataFrame()
    # print(files[0])
    for file in files:
        try:
            pti = str(file)[124:128]
            df_temp = pd.read_excel(str(file))
            df_temp = df_temp.iloc[:,0:2]
            df_temp = df_temp.rename(columns={df_temp.columns[0]: 'DateTime'})
            df_temp = df_temp.rename(columns={df_temp.columns[1]: 'GlucValue'})
            df_temp['PtID'] = pti
            df_temp['DateTime'] = pd.to_datetime(df_temp.DateTime, errors='coerce', infer_datetime_format=True)
            df_temp = df_temp[['PtID','DateTime','GlucValue']]
            df_temp = df_temp.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
            df_temp= df_temp.reset_index(drop=True)
            df_temp = df_temp.drop_duplicates(subset=['ts','id'])
        except Exception as e:
            print(f"Error al leer {file.name}: {e}")
        folder_dfs = pd.concat([folder_dfs, df_temp])
    return folder_dfs
Shanghai_T1D = Loading_File(T1D)
Shanghai_T2D = Loading_File(T2D)
Shanghai = pd.concat([Shanghai_T1D, Shanghai_T2D])
#pivoting the dataset
# data = Shanghai.pivot(index='ts', columns=['id'], values=['gl'])
# data.columns = data.columns.get_level_values(level='id')
Shanghai


,id,ts,gl
0,1001,2021-07-30 16:43:00,113.4
1,1001,2021-07-30 16:58:00,124.2
2,1001,2021-07-30 17:13:00,129.6
3,1001,2021-07-30 17:28:00,142.2
4,1001,2021-07-30 17:43:00,156.6
...,...,...,...
1324,2099,2020-11-30 07:47:00,97.2
1325,2099,2020-11-30 08:02:00,93.6
1326,2099,2020-11-30 08:17:00,90.0
1327,2099,2020-11-30 08:32:00,88.2


# Transform them into Sequences

## Sequences extracted from the notebook

In [168]:
from src.New_Utils import New_Sequences
seq = New_Sequences(Replace)
len(seq)

6829

In [167]:
from src.New_Utils import New_Sequences
seq = New_Sequences(AIDE)
len(seq)

174

In [166]:
from src.New_Utils import New_Sequences
seq = New_Sequences(Shanghai, minutes = 15)
len(seq)

45

## Sequences from the Datasets library

In [169]:
from Datasets import Replace
from Datasets import AIDE
from Datasets import Shanghai

In [170]:
from src.New_Utils import New_Sequences
seq = New_Sequences(Replace)
len(seq)

6829

In [171]:
from src.New_Utils import New_Sequences
seq = New_Sequences(AIDE)
len(seq)

174

In [173]:
from src.New_Utils import New_Sequences
seq = New_Sequences(Shanghai, minutes =15)
len(seq)

45